# Evaluation: property valuation accuracy and error analysis

This notebook evaluates the property assessment regression models by:

1. Predicted vs. actual scatter plot
2. Residual analysis
3. Error breakdown by property class
4. Feature importance with SHAP
5. R-squared 0.77 discussion and limitations
6. Valuation accuracy by neighbourhood

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sys.path.insert(0, '..')
from src.data_loader import load_or_fetch_data, preprocess_data, engineer_features
from src.model import (
    prepare_model_data,
    train_models,
    get_feature_importance,
    explain_prediction,
)

DATA_DIR = str(Path('..') / 'data')
pd.set_option('display.max_columns', 40)
print('Setup complete.')

In [ ]:
raw = load_or_fetch_data(DATA_DIR, limit=100000)
df = preprocess_data(raw)
df = engineer_features(df)
X, y, label_encoders, feature_names = prepare_model_data(df)

trained_models, results, scaler, X_test, y_test = train_models(X, y, random_state=42)

# Use the best model (XGBoost typically)
best_name = max(results, key=lambda k: results[k]['R2'])
best_model = trained_models[best_name]

if best_name == 'Ridge Regression':
    y_pred = best_model.predict(scaler.transform(X_test))
else:
    y_pred = best_model.predict(X_test)

y_pred_clipped = np.clip(y_pred, y.min(), y.max())

print(f'Best model: {best_name}')
print(f'R2: {r2_score(y_test, y_pred):.4f}')
print(f'Test samples: {len(y_test):,}')

## 1. Predicted vs. actual

A scatter plot of predicted vs. actual log-transformed values. Points close
to the diagonal line indicate accurate predictions.

In [ ]:
pred_df = pd.DataFrame({
    'Actual (log)': y_test.values,
    'Predicted (log)': y_pred_clipped,
})
pred_df['Actual ($)'] = np.expm1(pred_df['Actual (log)'])
pred_df['Predicted ($)'] = np.expm1(pred_df['Predicted (log)'])

fig = px.scatter(
    pred_df.sample(min(3000, len(pred_df)), random_state=42),
    x='Actual ($)', y='Predicted ($)',
    title=f'Predicted vs. actual property values ({best_name})',
    opacity=0.4,
)
max_val = max(pred_df['Actual ($)'].max(), pred_df['Predicted ($)'].max())
fig.add_trace(go.Scatter(x=[0, max_val], y=[0, max_val],
                         mode='lines', line=dict(dash='dash', color='red'),
                         name='Perfect prediction'))
fig.update_layout(height=500, width=550)
fig.show()

## 2. Residual analysis

Residuals (actual - predicted) should be centred at zero with no clear
pattern. Systematic deviations indicate model bias.

In [ ]:
pred_df['Residual ($)'] = pred_df['Actual ($)'] - pred_df['Predicted ($)']
pred_df['Residual (log)'] = pred_df['Actual (log)'] - pred_df['Predicted (log)']

fig = make_subplots(rows=1, cols=2, subplot_titles=['Residuals vs. predicted (log)', 'Residual distribution'])

sample = pred_df.sample(min(3000, len(pred_df)), random_state=42)
fig.add_trace(go.Scatter(x=sample['Predicted (log)'], y=sample['Residual (log)'],
                         mode='markers', marker=dict(opacity=0.3, color='steelblue'),
                         showlegend=False), row=1, col=1)
fig.add_hline(y=0, line_dash='dash', line_color='red', row=1, col=1)

fig.add_trace(go.Histogram(x=pred_df['Residual (log)'], nbinsx=50,
                           marker_color='darkorange', showlegend=False), row=1, col=2)

fig.update_layout(title='Residual analysis', height=400)
fig.show()

print(f'Mean residual (log): {pred_df["Residual (log)"].mean():.4f}')
print(f'Std residual (log):  {pred_df["Residual (log)"].std():.4f}')

## 3. Error by property class

Does the model perform differently for residential vs. non-residential
properties?

In [ ]:
# Recover property_class for test rows
if 'property_class' in df.columns and 'property_class' in label_encoders:
    le_pc = label_encoders['property_class']
    test_pc_encoded = X_test['property_class'].values if 'property_class' in X_test.columns else None
    if test_pc_encoded is not None:
        pred_df['property_class'] = le_pc.inverse_transform(test_pc_encoded.astype(int))
    else:
        pred_df['property_class'] = 'Unknown'
else:
    pred_df['property_class'] = 'Unknown'

error_by_class = pred_df.groupby('property_class').agg(
    count=('Residual ($)', 'count'),
    mae=('Residual ($)', lambda x: np.abs(x).mean()),
    median_error=('Residual ($)', 'median'),
    r2=('Actual (log)', lambda x: r2_score(
        pred_df.loc[x.index, 'Actual (log)'],
        pred_df.loc[x.index, 'Predicted (log)'],
    ) if len(x) > 1 else np.nan),
).reset_index()

error_by_class

In [ ]:
if pred_df['property_class'].nunique() > 1:
    fig = px.box(
        pred_df, x='property_class', y='Residual ($)',
        title='Residual distribution by property class',
        labels={'property_class': 'Property class', 'Residual ($)': 'Residual ($)'},
        color='property_class',
    )
    fig.update_layout(height=400, showlegend=False)
    fig.show()

## 4. Feature importance (SHAP)

SHAP (SHapley Additive exPlanations) provides a principled way to measure
each feature's contribution to individual predictions. We use
`explain_prediction` from `model.py`.

In [ ]:
# Try SHAP; fall back to native importance if unavailable
sample_X = X_test.head(500)
explainer, shap_values = explain_prediction(best_model, sample_X, feature_names)

if shap_values is not None:
    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    shap_df = pd.DataFrame({
        'Feature': feature_names,
        'Mean |SHAP|': mean_abs_shap,
    }).sort_values('Mean |SHAP|', ascending=False)

    fig = px.bar(shap_df, x='Mean |SHAP|', y='Feature', orientation='h',
                 title='SHAP feature importance (mean absolute SHAP value)',
                 color='Mean |SHAP|', color_continuous_scale='YlOrRd')
    fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height=400)
    fig.show()
    shap_df
else:
    print('SHAP not available; using native feature importance instead.')
    importance = get_feature_importance(best_model, feature_names, best_name)
    fig = px.bar(importance, x='Importance', y='Feature', orientation='h',
                 title=f'{best_name} native feature importance',
                 color='Importance', color_continuous_scale='YlOrRd')
    fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height=400)
    fig.show()
    importance

## 5. R-squared 0.77 discussion

An R-squared of approximately 0.77 means the model explains about 77% of the
variance in log-transformed property values. The remaining 23% is attributable
to factors not captured in our feature set.

In [ ]:
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(np.expm1(y_test), np.expm1(y_pred_clipped))
rmse = np.sqrt(mean_squared_error(np.expm1(y_test), np.expm1(y_pred_clipped)))

print(f'R-squared (log scale): {r2:.4f}')
print(f'MAE (original $):      ${mae:,.0f}')
print(f'RMSE (original $):     ${rmse:,.0f}')
print()
print('Factors not captured in the model that would improve accuracy:')
print('- Building size / square footage')
print('- Number of bedrooms and bathrooms')
print('- Lot size and frontage')
print('- Year built and renovation history')
print('- Proximity to transit, schools, and parks')
print('- Current market conditions and recent sales')

## 6. Valuation accuracy by neighbourhood

Some neighbourhoods are predicted more accurately than others, depending
on homogeneity of property values within the community.

In [ ]:
# Recover community labels for test rows
if 'community' in label_encoders:
    le_comm = label_encoders['community']
    if 'community' in X_test.columns:
        pred_df['community'] = le_comm.inverse_transform(X_test['community'].values.astype(int))
    else:
        pred_df['community'] = 'Unknown'
else:
    pred_df['community'] = 'Unknown'

community_error = pred_df.groupby('community').agg(
    count=('Residual ($)', 'count'),
    mae=('Residual ($)', lambda x: np.abs(x).mean()),
    mape=('Actual ($)', lambda x: (np.abs(pred_df.loc[x.index, 'Residual ($)']) / x.clip(lower=1)).mean() * 100),
).reset_index()
community_error = community_error[community_error['count'] >= 10]
community_error = community_error.sort_values('mae', ascending=True)

In [ ]:
# Best 10 and worst 10 communities by MAE
best_10 = community_error.head(10)
worst_10 = community_error.tail(10)

fig = make_subplots(rows=1, cols=2, subplot_titles=['Most accurate (lowest MAE)', 'Least accurate (highest MAE)'])

fig.add_trace(go.Bar(x=best_10['mae'], y=best_10['community'], orientation='h',
                     marker_color='mediumseagreen', showlegend=False), row=1, col=1)
fig.add_trace(go.Bar(x=worst_10['mae'], y=worst_10['community'], orientation='h',
                     marker_color='salmon', showlegend=False), row=1, col=2)

fig.update_layout(title='Valuation accuracy by neighbourhood (MAE)', height=450)
fig.update_yaxes(categoryorder='total ascending', row=1, col=1)
fig.update_yaxes(categoryorder='total ascending', row=1, col=2)
fig.show()

In [ ]:
print(f'\nBest-predicted communities (lowest MAE):')
print(best_10[['community', 'count', 'mae', 'mape']].to_string(index=False))
print(f'\nWorst-predicted communities (highest MAE):')
print(worst_10[['community', 'count', 'mae', 'mape']].to_string(index=False))

## Summary

Key evaluation findings:

1. The predicted vs. actual scatter plot shows strong alignment along the diagonal, confirming the model captures the general trend.
2. Residuals are approximately centred at zero with no obvious heteroscedasticity in log space.
3. Residential properties are predicted more accurately than non-residential ones, reflecting the larger training sample.
4. SHAP analysis confirms that community-level aggregate features (`community_avg_value`, `community_median_value`) dominate predictions.
5. The R-squared of approximately 0.77 is reasonable given the limited feature set; adding building-level attributes would likely improve it.
6. Homogeneous suburban neighbourhoods have lower MAE, while mixed-use downtown areas are harder to predict.

These results are surfaced in the interactive Streamlit dashboard (`app.py`).